In [10]:
import json
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import nltk
from nltk import RegexpParser
from nltk.corpus import wordnet as wn
from nltk.corpus import gutenberg, brown, reuters
from nltk.tokenize import word_tokenize
import spacy
from tqdm import tqdm

In [ ]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('gutenberg')
nltk.download('brown')
nltk.download('reuters')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nisan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\nisan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\nisan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nisan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\nisan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nisan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt 

True

In [12]:
nlp = spacy.load("en_core_web_sm")

OUT = Path("module3_outputs")
OUT.mkdir(exist_ok=True)

In [13]:
def extract_np_spacy(lines):
    results = []
    for i, line in enumerate(lines):
        doc = nlp(line)
        for np in doc.noun_chunks:
            results.append({'line_id': i, 'line': line, 'noun_phrase': np.text})
    return pd.DataFrame(results)

In [14]:
grammar = r"NP: {<DT>?<JJ>*<NN>+}"
chunker = RegexpParser(grammar)

def extract_np_nltk(pos_sentences):
    # pos_sentences: list of lists of (token, pos) pairs
    rows = []
    for sid, sent in enumerate(pos_sentences):
        tree = chunker.parse(sent)
        for subtree in tree.subtrees():
            if subtree.label() == 'NP':
                np_text = " ".join(w for w,t in subtree.leaves())
                rows.append({'sent_id': sid, 'noun_phrase': np_text})
    return pd.DataFrame(rows)

# corpus stats helper
def corpus_stats(words):
    total = len(words)
    vocab = set(words)
    freq = Counter(words)
    top30 = freq.most_common(30)
    return {'total_tokens': total,
            'vocab_size': len(vocab),
            'top_30': top30,
            'type_token_ratio': len(vocab)/total if total>0 else 0}

In [ ]:
def wordnet_info(word, pos=wn.NOUN):
    syns = wn.synsets(word, pos=pos)
    out = []
    for s in syns:
        out.append({
            'synset_name': s.name(),
            'definition': s.definition(),
            'examples': s.examples(),
            'lemmas': [l.name() for l in s.lemmas()],
            'antonyms': [ant.name() for l in s.lemmas() for ant in l.antonyms()],
            'hypernyms': [h.name() for h in s.hypernyms()],
            'hyponyms': [h.name() for h in s.hyponyms()],
            'meronyms': [m.name() for m in s.part_meronyms()+
                         s.substance_meronyms()+
                         s.member_meronyms()]
        })
    return out

In [16]:
lines = None

if Path("Shakespeare.csv").exists():
    df = pd.read_csv("Shakespeare.csv", dtype=str, keep_default_na=False)
    if 'PlayerLine' in df.columns:
        lines = df['PlayerLine'].astype(str).str.strip().tolist()

# normalize
lines = [l for l in lines if l and l.strip()]

df_np_spacy = extract_np_spacy(lines)
df_np_spacy.to_csv(OUT / "noun_phrases_spacy.csv", index=False, encoding='utf-8')

In [17]:
pos_sentences = []
if Path("pos_tagged.csv").exists():
    pos_df = pd.read_csv("pos_tagged.csv", dtype=str, keep_default_na=False)
    # attempt to group by line (column 'line' or 'line_id')
    if 'line' in pos_df.columns:
        grouped = pos_df.groupby('line')
        for _, g in grouped:
            sent = list(zip(g['token'].astype(str).tolist(), g['pos'].astype(str).tolist()))
            pos_sentences.append(sent)
    elif 'line_id' in pos_df.columns:
        grouped = pos_df.groupby('line_id')
        for _, g in grouped:
            sent = list(zip(g['token'].astype(str).tolist(), g['pos'].astype(str).tolist()))
            pos_sentences.append(sent)

In [20]:
if not pos_sentences:
    for l in lines:
        toks = word_tokenize(l)
        tagged = nltk.pos_tag(toks)  # returns (word, POS) tuples
        pos_sentences.append(tagged)

In [21]:
df_np_nltk = extract_np_nltk(pos_sentences)
df_np_nltk.to_csv(OUT / "noun_phrases_nltk.csv", index=False, encoding='utf-8')

In [22]:
guten_tokens = [w.lower() for w in gutenberg.words()]
brown_tokens = [w.lower() for w in brown.words()]
reuters_tokens = [w.lower() for w in reuters.words()]

stats = {
    'project_gutenberg': corpus_stats(guten_tokens),
    'brown_corpus': corpus_stats(brown_tokens),
    'reuters': corpus_stats(reuters_tokens)
}

In [24]:
rows = []
for name, s in stats.items():
    rows.append({'corpus': name, 'total_tokens': s['total_tokens'], 'vocab_size': s['vocab_size'],
                 'type_token_ratio': s['type_token_ratio'], 'top_word_1': s['top_30'][0][0] if s['top_30'] else ''})
pd.DataFrame(rows).to_csv(OUT / "corpus_stats.csv", index=False, encoding='utf-8')

In [25]:
for name, s in stats.items():
    pd.DataFrame(s['top_30'], columns=['word','count']).to_csv(OUT / f"top30_{name}.csv", index=False, encoding='utf-8')

# ---- F. WordNet analysis and similarity example ----
w = "processing"
wn_info = wordnet_info(w, pos=wn.NOUN)
with open(OUT / "wordnet_processing.json", "w", encoding='utf-8') as f:
    json.dump(wn_info, f, ensure_ascii=False, indent=2)

In [26]:
syn_car = wn.synsets('car', pos=wn.NOUN)
syn_auto = wn.synsets('automobile', pos=wn.NOUN)
best = 0.0
best_pair = (None, None)
for a in syn_car:
    for b in syn_auto:
        sim = a.wup_similarity(b) or 0.0
        if sim > best:
            best = sim
            best_pair = (a.name(), b.name())

with open(OUT / "wordnet_similarity_car_automobile.json", "w", encoding='utf-8') as f:
    json.dump({'best_pair': best_pair, 'wup_similarity': best}, f, ensure_ascii=False, indent=2)

In [27]:
print("Saved: noun_phrases_spacy.csv (rows: {})".format(len(df_np_spacy)))
print("Saved: noun_phrases_nltk.csv (rows: {})".format(len(df_np_nltk)))
print("Saved: corpus_stats.csv and top30_*.csv")
print("Saved: wordnet_processing.json and wordnet_similarity_car_automobile.json")
print("Output directory:", OUT.resolve())

Saved: noun_phrases_spacy.csv (rows: 262166)
Saved: noun_phrases_nltk.csv (rows: 133889)
Saved: corpus_stats.csv and top30_*.csv
Saved: wordnet_processing.json and wordnet_similarity_car_automobile.json
Output directory: C:\Users\nisan\Desktop\code\nlp\module3_outputs
